# Pilot solver runs on Colab

Fits the flow maxent solver over the committed LLM elicitation cache
(`metaculus/llm_cache_full.json`) — **no API keys needed**, the LLM half is
already done. Two configs: main and the `--no-corr` information-diet ablation.

Runtime guide: ~20-30 s/question on CPU, so 147 questions x 2 configs is
roughly 2-2.5 h on a single runtime. To split across k parallel Colab
sessions, give each one `--shard i/k` and its own `--out` file, then merge
with the last cell.

In [ ]:
!git clone https://github.com/amdson/calibrated_response.git
%cd calibrated_response
%pip install -q -e .

In [ ]:
# use the GPU when the runtime has one (the solver script defaults to CPU
# only when JAX_PLATFORMS is unset)
import os, subprocess
has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
os.environ["JAX_PLATFORMS"] = "cuda" if has_gpu else "cpu"
print("JAX_PLATFORMS =", os.environ["JAX_PLATFORMS"])

In [ ]:
# main config. Resume-safe: rerunning this cell skips finished entries.
# For parallel sessions: add  --shard 0/2  (and 1/2 in the other session,
# each with its own --out file).
!python metaculus/run_flow_solver.py --out metaculus/flow_predictions.json

In [ ]:
# ablation: same estimates minus the correlation constraints
!python metaculus/run_flow_solver.py --no-corr --out metaculus/flow_predictions_nocorr.json

In [ ]:
!python metaculus/pilot_diagnostics.py --predictions \
    metaculus/flow_predictions.json metaculus/flow_predictions_nocorr.json

In [ ]:
# get the results off the runtime (pick one)
from google.colab import files
files.download("metaculus/flow_predictions.json")
files.download("metaculus/flow_predictions_nocorr.json")
# ...or mount Drive and copy:
# from google.colab import drive; drive.mount('/content/drive')
# !cp metaculus/flow_predictions*.json /content/drive/MyDrive/

In [ ]:
# OPTIONAL: merge shard outputs from parallel sessions into one file
# import json, glob
# merged = {}
# for p in sorted(glob.glob("metaculus/flow_predictions_shard*.json")):
#     merged.update(json.load(open(p)))
# json.dump(merged, open("metaculus/flow_predictions.json", "w"), indent=1)
# print(len(merged), "rows merged")